##### Extracting Date Components
- You can use the specific functions to **extract components** from an existing **date or timestamp** column.

- **year(col):**
  - `Extracts the year as an integer`.

- **month(col):**
  - `Extracts the month as an integer`.

- **dayofmonth(col) or day(col):**
  - `Extracts the day of the month as an integer`.

In [0]:
from pyspark.sql.functions import col, lit, year, month, day, dayofmonth, count, sum, concat_ws

##### 1) Extracting Year, Month, Day from a Date Column

In [0]:

data = [("2026-02-16",), ("2025-12-31",), ("2025-01-01",), ("2025-01-10",), ("2025-01-28",)]

# Create a DataFrame with a date column
df_day = (spark.createDataFrame(data, ["order_date"])
    # Convert the date column to a DateType
    .withColumn("order_date", col("order_date").cast("date")))

display(df_day)

order_date
2026-02-16
2025-12-31
2025-01-01
2025-01-10
2025-01-28


##### Using year(), month(), day()

**Method 01**

     from pyspark.sql.functions import year, month, day, dayofmonth

     df_extracted = df.withColumn("year", year(col("order_date"))) \
                      .withColumn("month", month(col("order_date"))) \
                      .withColumn("day", day(col("order_date"))) \                 # dayofmonth(col) or day(col)
                      .withColumn("dayofmonth", dayofmonth(col("order_date")))     # dayofmonth(col) or day(col)

**Method 02:**

     from pyspark.sql import functions as F

     # Extracting year, month, and day into new columns
     df_extracted = df.withColumn("birth_year", F.year(F.col("birth_date"))) \
                      .withColumn("birth_month", F.month(F.col("birth_date"))) \
                      .withColumn("birth_day", F.day(F.col("birth_date")))            # dayofmonth(col) or day(col)
                      .withColumn("birth_day", F.dayofmonth(F.col("birth_date")))     # dayofmonth(col) or day(col)

In [0]:
# Extracting year, month and day into new columns
df_extracted = df_day \
    .withColumn("year", year(col("order_date"))) \
    .withColumn("month", month(col("order_date"))) \
    .withColumn("day", day(col("order_date"))) \
    .withColumn("dayofmonth", dayofmonth(col("order_date")))

display(df_extracted)

order_date,year,month,day,dayofmonth
2026-02-16,2026,2,16,16
2025-12-31,2025,12,31,31
2025-01-01,2025,1,1,1
2025-01-10,2025,1,10,10
2025-01-28,2025,1,28,28


In [0]:
%sql
SELECT
    current_date() AS today,
    year(current_date()) AS year,
    month(current_date()) AS month,
    day(current_date()) AS day,
    weekofyear(current_date()) AS week_of_year,
    quarter(current_date()) AS quarter;

today,year,month,day,week_of_year,quarter
2026-02-21,2026,2,21,8,1


In [0]:
from pyspark.sql.functions import current_date, year, month, day, weekofyear, quarter

In [0]:
df = spark.createDataFrame([(1,)], ["dummy"])
display(df)

df.select(
    current_date().alias("today"),
    year(current_date()).alias("year"),
    month(current_date()).alias("month"),
    day(current_date()).alias("day"),
    weekofyear(current_date()).alias("week_of_year"),
    quarter(current_date()).alias("quarter")
).display()

dummy
1


today,year,month,day,week_of_year,quarter
2026-03-09,2026,3,9,11,1


In [0]:
df = spark.range(1)
display(df)

df.select(
    current_date().alias("today"),
    year(current_date()).alias("year"),
    month(current_date()).alias("month"),
    day(current_date()).alias("day"),
    weekofyear(current_date()).alias("week_of_year"),
    quarter(current_date()).alias("quarter")
).display()

id
0


today,year,month,day,week_of_year,quarter
2026-03-09,2026,3,9,11,1


##### 2) How to Create a New Date Column Using Year, Month, Day

In [0]:
# Create a DataFrame with a date column
data = [(1, "2022-01-01"), (2, "2022-02-01"), (3, "2022-03-01"), (4, "2022-04-01"), (5, "2022-05-01")]

df_new = spark.createDataFrame(data, ["id", "date"])
# Convert the date column to a DateType
df_new = df_new.withColumn("date", col("date").cast("date"))

final_df_new = df_new.withColumn(
    "new_date",
    concat_ws(
        "-",
        (year(col("date")) + 1),
        (month(col("date")) + 1),
        (dayofmonth(col("date")) + 1),
    )
)

display(final_df_new)

id,date,new_date
1,2022-01-01,2023-2-2
2,2022-02-01,2023-3-2
3,2022-03-01,2023-4-2
4,2022-04-01,2023-5-2
5,2022-05-01,2023-6-2


##### 3) Group Data Using Year, Month, Day

In [0]:
# Create a DataFrame with a date column
data = [(1, "2022-01-01", 100),
        (2, "2022-01-01", 250),
        (3, "2023-01-02", 200),
        (4, "2023-01-02", 250),
        (5, "2023-01-02", 150),
        (6, "2023-01-03", 150),
        (7, "2024-02-01", 300),
        (8, "2024-02-01", 200),
        (9, "2025-05-09", 500),
        (10, "2026-02-02", 150),
        (11, "2026-02-02", 250)]

sales_df = spark.createDataFrame(data, ["id", "transaction_date", "amount"])

# Convert the date column to a DateType
sales_df = sales_df.withColumn("transaction_date", sales_df["transaction_date"].cast("date"))
display(sales_df)

id,transaction_date,amount
1,2022-01-01,100
2,2022-01-01,250
3,2023-01-02,200
4,2023-01-02,250
5,2023-01-02,150
6,2023-01-03,150
7,2024-02-01,300
8,2024-02-01,200
9,2025-05-09,500
10,2026-02-02,150


In [0]:
from pyspark.sql.functions import year, month, dayofmonth, sum, count

In [0]:
# Group data by year, month, day and count the number of rows
sales_df_final = sales_df.groupBy(
    year("transaction_date").alias("year"),
    month("transaction_date").alias("month"),
    dayofmonth("transaction_date").alias("day")
    ).agg(
        count("*").alias("count"),
        sum("amount").alias("total_sales"))

# Display the result
display(sales_df_final)

year,month,day,count,total_sales
2022,1,1,2,350
2023,1,2,3,600
2023,1,3,1,150
2024,2,1,2,500
2025,5,9,1,500
2026,2,2,2,400


In [0]:
from pyspark.sql.functions import year, month, dayofmonth

final_df = sales_df \
    .withColumn("year", year("transaction_date")) \
    .withColumn("month", month("transaction_date")) \
    .withColumn("day", dayofmonth("transaction_date"))

final_df = final_df.groupBy("year", "month", "day") \
                   .agg(
                       count("*").alias("count"),
                       sum("amount").alias("total_sales"))
                   
display(final_df)

year,month,day,count,total_sales
2022,1,1,2,350
2023,1,2,3,600
2023,1,3,1,150
2024,2,1,2,500
2025,5,9,1,500
2026,2,2,2,400


##### 4) Group by Full Date
- If you only want `daily grouping`.

In [0]:
from pyspark.sql.functions import to_date

sales_df_sgle = sales_df.groupBy("transaction_date") \
    .agg(
        count("*").alias("count"),
        sum("amount").alias("total_sales"))
    
display(sales_df_sgle)

transaction_date,count,total_sales
2022-01-01,2,350
2023-01-02,3,600
2023-01-03,1,150
2024-02-01,2,500
2025-05-09,1,500
2026-02-02,2,400
